# SCENE — 一键运行的 MuJoCo 场景与任务集

## 为什么要这本 notebook

**问题**：自己手搓的 XML 场景（自定义木桌、随机摆物、自选机械臂）通常**不在任何模型的训练分布里**。VLA / RL 模型在自定义场景上跑出来的行为，多半只是「策略看到没见过的画面在随机抽搐」。

**解决**：直接复用社区里**已经被广泛使用、模型训练过、有公开数据集**的现成场景。下面这些库要么 `pip install` 一行，要么 `git clone` 一句，开箱就能 `env.reset() / env.step()`，渲染器、相机、物体、机器人统统打包好。

## 怎么分级

| 级别 | 说明 | 代表 |
| --- | --- | --- |
| 🟢 **一线（真一键）** | 一行 `pip install`，资产 < 500 MB | dm_control, gymnasium-robotics, robosuite, metaworld, mjlab, mujoco_playground |
| 🟡 **重资产** | 安装本身轻，但**数据/资产几个 GB 起**，注意磁盘 | RoboCasa (~10 GB), LIBERO (~3 GB), MimicGen 数据集 |
| 🔵 **本仓已有** | 通过 git submodule 已经在 repo 里 | `dependencies/mujoco_menagerie/*/scene.xml` |
| ⚫ **避坑** | 名字常被误传，实际不可用 | ManiSkill (是 SAPIEN 不是 MuJoCo)，D4RL (已弃用)，Open X-Embodiment 的 MuJoCo 版本（不存在） |

## 渲染方式：实时原生窗口

每个 cell 跑起来会**弹出一个独立的 MuJoCo 原生窗口**（GLFW），就是 `./build/bin/simulate` 那种窗口：实时仿真、鼠标拖拽旋转、滚轮缩放、按 ESC / 关窗结束。不是 inline 视频，也不需要 mediapy。

> **前置**：必须有 X11 / Wayland 显示。SSH 远程要 `-X`/`-Y` 转发；纯 headless 服务器跑不了实时窗口，需要换成离屏录帧（每节末尾说明）。
> **环境**：`conda activate mujoco`。无 GPU 也能跑 90% 内容（`mujoco_playground` 训练才需要 JAX GPU）。

## 版本前置（重要）

下面这些场景库对 `mujoco` / `gymnasium` 版本有较新要求。本仓 `init.sh` 创建的 conda 环境可能停留在 `mujoco 3.3.x` / `gymnasium 0.29.x`，跑某些 cell 会触发 `DeprecatedEnv` 或缺类（如 `PandaOmron`）。建议**先升级一次**再开始：

```bash
pip install -U 'mujoco>=3.8.1' 'gymnasium>=1.2.0'
```

各库实际要求：

| 库 | 最低 mujoco | 最低 gymnasium |
| --- | --- | --- |
| dm_control 1.0.41 | 3.8.1 | — |
| gymnasium-robotics 1.4.2 | — | 1.2.0 |
| metaworld 3.0.0 | — | 1.1 |
| mjlab 1.3.0 | 3.7.0 | — |
| playground 0.1.0 (mujoco_playground) | 3.4 | — |
| mujoco_mjx / mujoco_warp 3.8.1 | 3.8.0 | — |
| robosuite 1.5.2 | 3.3 | — (自家 API) |

> 如果 `pip install -U mujoco` 跟其他 pin 冲突，最干净的做法是**给场景调研新开一个 conda 环境**：`conda create -n scenes python=3.10 && conda activate scenes && pip install mujoco dm_control gymnasium-robotics ...`。避免污染 `mujoco` 主环境。

---

## 🟢 一线：真一键

下面六个库是「装完即用」的，每个都给：① 安装命令，② 启动一个代表场景的最短 Python 片段。

### 1. dm_control — DeepMind 经典控制套件

- **出身**：DeepMind 官方，MuJoCo 的「亲儿子」之一。
- **包含**：`suite`（cartpole / walker / cheetah / humanoid / quadruped / manipulator …），`mjcf`（程序化拼场景），`viewer`（交互窗口）。
- **资产**：约 50 MB，全部打包在 wheel 内。
- **API**：返回 `TimeStep`（不是 Gymnasium）。

In [ ]:
!pip install -q dm_control

In [ ]:
# 启动一个 humanoid stand 任务并交互渲染（按 ESC 退出）。
from dm_control import suite, viewer
import numpy as np
env = suite.load(domain_name='humanoid', task_name='stand')
action_spec = env.action_spec()
def random_policy(_):
    return np.random.uniform(action_spec.minimum, action_spec.maximum, size=action_spec.shape)
viewer.launch(env, policy=random_policy)

### 2. mujoco_playground — DeepMind 新官方 MJX 任务集

- **出身**：DeepMind 2025 年新开的 [google-deepmind/mujoco_playground](https://github.com/google-deepmind/mujoco_playground)，配套 MuJoCo Warp。
- **包含**：dm_control 经典任务移植 + locomotion（Unitree G1/Go1/双足）+ 非抓握/灵巧操作 + 视觉版渲染。
- **资产**：代码几十 MB，机器人资产首次使用从 `mujoco_menagerie` 懒加载（已有 submodule 可复用）。
- **注意**：**PyPI 包名叫 `playground`，不是 `mujoco_playground`**；GPU + CUDA 12 JAX 强烈推荐（CPU 也能跑只是慢）。

In [ ]:
!pip install -q playground

In [ ]:
# 载入 G1 平地 joystick 任务。env 本身是 JAX/MJX 的（适合做训练），
# 但 env.mj_model 暴露了底层 MuJoCo XML，我们直接用 mujoco.viewer 实时预览场景。
import mujoco, mujoco.viewer, time
from mujoco_playground import registry
env = registry.load('G1JoystickFlatTerrain')
print(f'action_size={env.action_size}  observation_size={env.observation_size}')
model = env.mj_model
data = mujoco.MjData(model)
with mujoco.viewer.launch_passive(model, data) as viewer:
    t0 = time.time()
    while viewer.is_running() and time.time() - t0 < 15:
        mujoco.mj_step(model, data)
        viewer.sync()

### 3. gymnasium-robotics — Farama Fetch / Shadow Hand

- **出身**：Farama Foundation 维护，gymnasium 标准化生态。
- **包含**：`FetchReach/Push/Slide/PickAndPlace`，`HandManipulate*`（Shadow Hand），`AntMaze`，`PointMaze`。
- **资产**：约 30 MB。
- **API**：标准 Gymnasium。

In [ ]:
!pip install -q gymnasium-robotics

In [ ]:
# FetchPickAndPlace：经典桌面抓取-放置任务。
# 注意：gymnasium-robotics 已跳到 v4（v2/v3 已 deprecated）。
import gymnasium as gym
import gymnasium_robotics  # noqa: F401  (registers envs)
gym.register_envs(gymnasium_robotics)
env = gym.make('FetchPickAndPlace-v4', render_mode='human')
obs, _ = env.reset(seed=0)
for _ in range(500):
    action = env.action_space.sample()
    obs, r, term, trunc, _ = env.step(action)
    if term or trunc:
        obs, _ = env.reset()
env.close()

### 4. robosuite — Panda/Sawyer 经典桌面操作

- **出身**：ARISE Initiative（UT Austin + Stanford），机器人操作研究的事实标准之一。
- **包含**：Lift / Stack / NutAssembly / PickPlace / Door / Wipe / TwoArm* 等，机械臂支持 Panda / Sawyer / UR5e / Kinova3 / IIWA，v1.5 起增加人形与双臂。
- **资产**：约 100 MB。
- **API**：自家 API（`env.action_spec` 是 `(low, high)` 元组，不是 Gymnasium）。

In [ ]:
!pip install -q robosuite

In [ ]:
# Lift：Panda 抓起桌面立方体的入门任务。
import numpy as np
import robosuite as suite
env = suite.make(env_name='Lift', robots='Panda',
                 has_renderer=True, has_offscreen_renderer=False,
                 use_camera_obs=False, control_freq=20)
low, high = env.action_spec
env.reset()
for _ in range(500):
    a = np.random.uniform(low, high)
    env.step(a)
    env.render()
env.close()

### 5. Meta-World — Sawyer 50 任务套件

- **出身**：现归 Farama Foundation 维护。
- **包含**：50 个桌面操作任务（reach / push / pick-place / button / door / drawer …），常用基准 MT1 / MT10 / MT50 / ML1 / ML45。
- **资产**：约 50 MB。
- **API**：标准 Gymnasium。

In [ ]:
!pip install -q metaworld

In [ ]:
# pick-place：从桌上某点拿起方块放到目标点。
import gymnasium as gym
import metaworld  # noqa: F401
env = gym.make('Meta-World/MT1', env_name='pick-place-v3', render_mode='human')
obs, _ = env.reset(seed=0)
for _ in range(500):
    a = env.action_space.sample()
    obs, r, term, trunc, _ = env.step(a)
    if term or trunc:
        obs, _ = env.reset()
env.close()

### 6. mjlab — Isaac-Lab-style on MuJoCo Warp

- **出身**：[mujocolab/mjlab](https://github.com/mujocolab/mjlab)（UC Berkeley，Kevin Zakka 等，2026 春）。**不是** Google DeepMind，名字别搞混。
- **包含**：Isaac Lab 风格的 manager API，开箱即用 Unitree G1 / Go1 / Go2 + 速度跟踪 / 运动模仿 / 操作任务。
- **资产**：核心代码很小，机器人懒加载自 menagerie。
- **注意**：训练需要 NVIDIA GPU；macOS 仅评估。

上面用到的 `diasAiMaster/unitree-go2-velocity-flat` ONNX 策略，就是用 mjlab 训出来的（详见 [`scripts/quadruped_locomotion_demo.py`](./scripts/quadruped_locomotion_demo.py)）。

In [ ]:
# 不想装也可以直接用 uvx 跑官方 demo（无需 pip install）：
#   uvx --from mjlab --refresh demo
!pip install -q mjlab

In [ ]:
# mjlab 装完后会注入几个 console_scripts：demo / play / train / list-envs。
# 注意是裸命令，不是 `mjlab demo`。
!list-envs

# 启动官方 demo 窗口（关掉窗口或 Ctrl+C 退出）：
!demo

---

## 🟡 重资产：装包轻，数据重

下面这几个**代码本身好装**，但**配套资产/数据集都是 GB 起跳**，先看清磁盘和带宽再下。所有这些库都基于 robosuite，所以建议先把 robosuite 装好。

### 7. RoboCasa — NVIDIA 厨房场景

- **出身**：UT Austin + NVIDIA。
- **包含**：100+ 厨房子场景（柜台、冰箱、橱柜、水池…）+ PickPlace / Open / Close / Turn / Press 等原子任务，用 robosuite + Panda / GR-1 humanoid。
- **资产**：**约 10 GB 厨房贴图与几何**，需要单独跑下载脚本。
- **依赖冲突**：RoboCasa setup.py 硬钉 `mujoco==3.3.1` / `numpy==2.2.5`，**和本 env 里其它场景库都冲突**。

👉 **已单独开页**：[`RoboCasa.ipynb`](./RoboCasa.ipynb) — 独立 conda env `robocasa`，从安装到运行一键化，包含 10+ 厨房场景 / 任务 / 机器人 demo。本 notebook 不再尝试装 RoboCasa。


### 8. LIBERO — VLA 终身学习基准

- **出身**：UT Austin Lifelong-Robot-Learning 组，发布 LIBERO-Spatial/Object/Goal/100 四套，**130 个任务**。
- **包含**：每个任务一个 BDDL 文件 + robosuite 子场景，Franka Panda；演示数据集（约 3 GB）可用于 BC/VLA 微调。
- **资产**：代码 ~200 MB；**数据集 multi-GB**（HF 上有）。
- **注意**：requirements 钉死 torch 1.11+cu113，大多数人会忽略它装现代 torch。

In [ ]:
# LIBERO 安装：clone -> 补一个 libero/__init__.py（仓库自身漏掉，否则 editable install 找不到包）-> editable 安装
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO /tmp/LIBERO 2>/dev/null || echo 'already cloned'
!touch /tmp/LIBERO/libero/__init__.py
!pip install -q -e /tmp/LIBERO --no-deps
!pip install -q bddl easydict

# 配置 ~/.libero/config.yaml 让 LIBERO 找得到 bddl/assets。
import os, yaml
os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
cfg = {
    'benchmark_root': '/tmp/LIBERO/libero/libero',
    'bddl_files': '/tmp/LIBERO/libero/libero/bddl_files',
    'init_states': '/tmp/LIBERO/libero/libero/init_files',
    'assets': '/tmp/LIBERO/libero/libero/assets',
    'datasets': '/tmp/LIBERO/datasets',
}
with open(os.path.expanduser('~/.libero/config.yaml'), 'w') as f:
    yaml.safe_dump(cfg, f)
print('libero config written')

In [ ]:
# LIBERO 设计为「离屏渲染喂 VLA 评测」，没有原生 onscreen viewer。
# 跑一次 reset 拿张 agentview 图直接 inline 显示。
import os, numpy as np
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from IPython.display import display
from PIL import Image

suite = benchmark.get_benchmark_dict()['libero_10']()
task = suite.get_task(0)
# task.bddl_file 只是 basename，必须拼上 bddl_files root + problem_folder。
bddl_path = os.path.join(get_libero_path('bddl_files'), task.problem_folder, task.bddl_file)
print('task:', task.language)

env = OffScreenRenderEnv(bddl_file_name=bddl_path, camera_heights=256, camera_widths=256)
obs = env.reset()
print('agentview_image shape:', obs['agentview_image'].shape)
display(Image.fromarray(np.flipud(obs['agentview_image'])))
env.close()

### 9. MimicGen — NVIDIA 数据增广套件

- **出身**：NVlabs，基于 robosuite 的演示生成器。
- **包含**：12 个核心任务，能从少量人类演示自动派生数万条变体轨迹；HF 上有**48k 预生成轨迹**（数十 GB）。
- **适合**：搞模仿学习/数据扩增的用户。日常体验 robosuite 就够了。
- **依赖链**：robosuite → robomimic → mimicgen，需依次装。

In [ ]:
!pip install -q robosuite robomimic
!git clone https://github.com/NVlabs/mimicgen /tmp/mimicgen 2>/dev/null || echo 'already cloned'
!pip install -q -e /tmp/mimicgen

---

## 🔵 本仓已有：menagerie 现成场景

`mujoco_menagerie` 是 DeepMind 维护的机器人模型库，本仓已作为 submodule 拉好。每个机器人目录下都有一个 `scene.xml`，可以直接用 C++ `simulate` 或 Python `viewer.launch` 打开预览。

In [ ]:
# 列出 menagerie 里所有自带 scene.xml 的机器人目录：
import os
root = 'dependencies/mujoco_menagerie'
robots = sorted(d for d in os.listdir(root)
                if os.path.isfile(os.path.join(root, d, 'scene.xml')))
print(f'共 {len(robots)} 个机器人带 scene.xml：')
for r in robots:
    print(' ', r)

In [ ]:
# 任意挑一个开窗预览（passive viewer，无策略，纯看场景）：
import mujoco, mujoco.viewer, time
xml_path = 'dependencies/mujoco_menagerie/unitree_go2/scene.xml'  # 换成上面列表里任意一个
model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)
with mujoco.viewer.launch_passive(model, data) as viewer:
    t0 = time.time()
    while viewer.is_running() and time.time() - t0 < 10:
        mujoco.mj_step(model, data)
        viewer.sync()

---

## ⚫ 别浪费时间：常见误传

下面这些经常被 *recommended as "MuJoCo scenes"*，**实际不是或不能用**：

| 名字 | 真相 |
| --- | --- |
| **ManiSkill 3** | 后端是 **SAPIEN**（不是 MuJoCo），尽管早期 ManiSkill 2 有 MuJoCo 实验性支持，v3 已完全脱离。要用就老老实实装 SAPIEN。 |
| **D4RL** | 已**官方弃用**。在线 env 迁到 `gymnasium-robotics`，离线数据集迁到 [Minari](https://github.com/Farama-Foundation/Minari)。 |
| **mj_envs** | 已被 [RoboHive](https://github.com/vikashplus/robohive) 吸收，单独装没意义。 |
| **Open X-Embodiment / BridgeData V2 "MuJoCo 版"** | **不存在官方 MuJoCo 移植**。OXE/BridgeV2 是真机轨迹数据集（RLDS），没有配套仿真场景。社区有零散 reconstruct（如 `jeongeun980906/Lerobot-MujoCo-VLA-Tutorial`）但远不完整。想跑 VLA 训练分布最近似的方案：用 LIBERO 或 RoboCasa。 |
| **"google-deepmind/mjlab"** | URL 是 404 的。正确仓库是 [`mujocolab/mjlab`](https://github.com/mujocolab/mjlab)，UC Berkeley 团队。 |

---

## 选型建议

- **想立刻看到画面、不挑任务**：dm_control → `humanoid stand` 或 `cheetah run`。
- **VLA 训练分布最近似**：LIBERO（130 任务 + 演示数据集，被 OpenVLA / RDT / π0 等模型用作评测）。
- **大规模厨房场景**：RoboCasa（需 10 GB 资产）。
- **机械臂 RL 基线**：robosuite（Lift / NutAssembly）或 metaworld（MT10 / MT50）。
- **四足 / 人形 locomotion**：mujoco_playground（DeepMind 官方 MJX）+ mjlab（Isaac-Lab 风格 + MuJoCo Warp）。
- **快速做手部灵巧操作**：gymnasium-robotics 里的 `HandManipulate*`。

## 下一步

选定哪个场景库后，可以把训练好的模型（OpenVLA / RDT / π0 / Octo / ACT）接到对应环境的相机观察 + 动作空间上，直接评测。我们 [`SCRIPTS.ipynb`](./SCRIPTS.ipynb) 里已经把这些模型在自定义场景下的最小工作流写好了，搬到 LIBERO / RoboCasa 主要替换 obs 字典和 action 缩放即可。